In [1]:
import pandas as pd
import sys
from pathlib import Path 
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import numpy as np

sys.path.insert(0, str(Path("../src").resolve()))
from idx_exchange_data_science.preprocessing import initial_cleaning, clean_living_area, select_training_window,\
    filter_close_price, clean_lot_size, clean_coordinates, CoordinateImputer, add_property_age_at_sale,\
    clean_property_characteristics, clean_postal_code, add_binary_missing_indicators, clean_categorical_features
    
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor

## Advanced Models: Decision Tree and Random Forest Regressors

This notebook develops and evaluates Decision Tree and Random Forest regression models to predict residential property closing prices (ClosePrice) using historical California real estate data. Their performance is compared against the baseline Linear Regression model.

### Data Loading and Preprocessing

The dataset consists of monthly CSV files from the California Regional Multiple Listing Service (CRMLS), covering January 2024 through April 2026.

The data is split chronologically into training, validation (March 2026), and test (April 2026) sets. Reusable cleaning functions from the idx_exchange.preprocessing_data_science module are applied to prepare the datasets for modeling.

Preprocessing parameters are learned from the training data, with fold-specific fitting used during cross-validation to prevent data leakage.

In [2]:
# Create a Path object to the folder containing the monthly CSV files
data_path = Path("../data")

# Get all the CSV files from the data folder and sort them chronologically by filename
csv_files = sorted(data_path.glob("*.csv"))

In [3]:
# Split the monthly CSV files into training, validation, and test sets
train_files = csv_files[:-2]
validation_file = csv_files[-2]
test_file = csv_files[-1]

In [4]:
# Read each monthly training file into a DataFrame
train_dfs = []

for file in train_files:
    monthly_data = pd.read_csv(file)

    # Extract the year and month from the CSV filename
    source_month = file.stem.replace("CRMLSSold", "").replace("_filled", "")
    monthly_data["SourceMonth"] = source_month[:4] + "-" + source_month[4:6]
    
    train_dfs.append(monthly_data)
    
# Combine the monthly training DataFrames into a single DataFrame
initial_data = pd.concat(train_dfs, ignore_index=True)

/var/folders/s8/2hb9crtx0f59zm5tv6jsyvhm0000gn/T/ipykernel_11222/84281947.py:5: DtypeWarning: Columns (2,36,39,56,74) have mixed types. Specify dtype option on import or set low_memory=False.
  monthly_data = pd.read_csv(file)
/var/folders/s8/2hb9crtx0f59zm5tv6jsyvhm0000gn/T/ipykernel_11222/84281947.py:5: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  monthly_data = pd.read_csv(file)
/var/folders/s8/2hb9crtx0f59zm5tv6jsyvhm0000gn/T/ipykernel_11222/84281947.py:5: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  monthly_data = pd.read_csv(file)


In [5]:
# Load validation and test files
validation_df = pd.read_csv(validation_file)
test_df = pd.read_csv(test_file)

# Add source month to each dataset
for df, file in [(validation_df, validation_file), (test_df, test_file)]:
    source_month = file.stem.replace("CRMLSSold", "").replace("_filled", "")
    df["SourceMonth"] = source_month[:4] + "-" + source_month[4:6]

### Initial Data Cleaning

Apply the initial cleaning rules to remove invalid records, including non-California properties, duplicate transactions, non-single-family residences, inconsistent dates, and invalid closing prices. The same cleaning rules are applied to all three datasets.

In [6]:
# Apply the initial cleaning rules to all datasets
clean_train_df, train_log = initial_cleaning(initial_data)
clean_validation_df, validation_log = initial_cleaning(validation_df)
clean_test_df, test_log = initial_cleaning(test_df)

In [7]:
display(train_log)

,step,Before,Removed,Removed (%),Remaining
0,California filtering,568074,76,0.0134,567998
1,Duplicate removal,567998,2911,0.5125,565087
2,Property-type filtering,565087,280370,49.6154,284717
3,Date consistency,284717,39,0.0137,284678
4,Target cleaning,284678,3,0.0011,284675


### Feature Selection

Select the relevant property features, target variable (ClosePrice), and tracking columns required for preprocessing and modeling. 

Features with 80% or more missing values are excluded from modeling. Based on the full 26 months training dataset, `BasementYN`, `BuildingAreaTotal`, and school-related columns exceed this threshold. 

Note: Detailed geographic layer using school districts will be added in Week 6 (Feature engineering). 

In [8]:
feature_columns = ["AttachedGarageYN", "BathroomsTotalInteger", "BedroomsTotal",
                   "City", "CountyOrParish", "FireplaceYN", "GarageSpaces", "Latitude",
                   "Longitude", "LivingArea","LotSizeSquareFeet", "LotSizeAcres", "NewConstructionYN",
                   "ParkingTotal", "PoolPrivateYN", "PostalCode", "ViewYN", "YearBuilt"]

target_column = ["ClosePrice"]
tracking_columns = ["SourceMonth", "CloseDate"]

key_columns = feature_columns + target_column + tracking_columns


In [9]:
clean_train_df = clean_train_df[key_columns].copy()
clean_validation_df = clean_validation_df[key_columns].copy()
clean_test_df = clean_test_df[key_columns].copy()

In [10]:
# Perserve the datasets before further preprocessing
base_train_df = clean_train_df.copy()
base_validation_df = clean_validation_df.copy()
base_test_df = clean_test_df.copy()

### Training Window Selection and Target Filtering

Select an adjustable training window of X months preceding March 2026. Calculate the 0.5th and 99.5th percentile ClosePrice thresholds from the selected training data and apply the same thresholds to the datasets according to the agreed cleaning rules.

In [11]:
X = 26  # Try 6, 12, 18

# Selcet training window 
clean_train_df = select_training_window(base_train_df, n_months=X, validation_month="2026-03")

print(f"Training window: {X} months")
print(f"Training records: {len(clean_train_df)}")

Training window: 26 months
Training records: 284675


In [12]:
# Calculate thresholds from the selected X-month training window
lower_threshold = clean_train_df["ClosePrice"].quantile(0.005)
upper_threshold = clean_train_df["ClosePrice"].quantile(0.995)

# Remove records outside the training-derived ClosePrice thresholds
clean_train_df, train_price_log = filter_close_price(clean_train_df, lower_threshold, upper_threshold)
clean_validation_df, validation_price_log = filter_close_price(clean_validation_df, lower_threshold, upper_threshold)
clean_test_df, test_price_log = filter_close_price(clean_test_df, lower_threshold, upper_threshold)

In [13]:
# # Apply log transformation to ClosePrice for all datasets
for df in[clean_train_df, clean_validation_df, clean_test_df]:
    df["LogClosePrice"] = np.log1p(df["ClosePrice"])

### Numerical Feature Cleaning

#### LivingArea and LotSizeSquareFeet

Clean numerical property features by removing records with nonpositive LivingArea values while retaining missing values for later imputation. Standardize lot-size measurements by converting valid acreage values to square feet when LotSizeSquareFeet is missing.

In [14]:
# Remove nonpositive LivingArea values while retaining missing values
clean_train_df, train_area_log = clean_living_area(clean_train_df)
clean_validation_df, validation_log = clean_living_area(clean_validation_df)
clean_test_df, test_log = clean_living_area(clean_test_df)

In [15]:
# Clean lot sizes and recover missing square footage using acres
clean_train_df, train_recovered_rows = clean_lot_size(clean_train_df)
clean_validation_df, validation_recovered_rows = clean_lot_size(clean_validation_df)
clean_test_df, test_recovered_rows = clean_lot_size(clean_test_df)

In [16]:
# Summarize the number of recovered lot-size values in each dataset
display(pd.DataFrame({
    "Dataset": ["Training", "Validation", "Test"],
    "Lot sizes recovered": [train_recovered_rows, validation_recovered_rows, test_recovered_rows]}))

,Dataset,Lot sizes recovered
0,Training,18
1,Validation,2
2,Test,0


#### Property Characteristics 

Property characteristics, including bedrooms, bathrooms, garage spaces, and parking counts, are evaluated using provisional plausibility ranges. Invalid values are marked as missing for later imputation rather than removing the corresponding records.

These cleaning rules have not yet been finalized and may be revised following team discussion and further review.

In [17]:
# Clean property characteristics and record invalid values
clean_train_df, train_property_log = clean_property_characteristics(clean_train_df)
clean_validation_df, validation_property_log = clean_property_characteristics(clean_validation_df)
clean_test_df, test_property_log = clean_property_characteristics(clean_test_df)

In [18]:
# Summarize invalid property characteristic values
property_logs = pd.DataFrame({ "Training": train_property_log,
                               "Validation": validation_property_log,
                               "Test": test_property_log})

display(property_logs)

,Training,Validation,Test
BedroomsTotal,90,6,2
BathroomsTotalInteger,65,5,3
GarageSpaces,537,19,21
ParkingTotal,1089,45,38


### Geographic Feature Cleaning

Standardize postal codes to five-digit ZIP codes and mark latitude and longitude values outside California's geographic bounds as missing. Missing coordinates will be imputed later using training-derived ZIP-level, county-level, and overall medians within the modeling pipeline.

In [19]:
# Standardize postal codes for all datasets
clean_train_df = clean_postal_code(clean_train_df)
clean_validation_df = clean_postal_code(clean_validation_df)
clean_test_df = clean_postal_code(clean_test_df)

In [20]:
# Mark invalid coordinates as missing
clean_train_df = clean_coordinates(clean_train_df)
clean_validation_df = clean_coordinates(clean_validation_df)
clean_test_df = clean_coordinates(clean_test_df)

### Feature Engineering

Calculate property age and create missingness indicators for binary property features. Missing values are retained for imputation within the modeling pipeline.

In [21]:
# Calculate property age at sale for all datasets
clean_train_df = add_property_age_at_sale(clean_train_df)
clean_validation_df = add_property_age_at_sale(clean_validation_df)
clean_test_df = add_property_age_at_sale(clean_test_df)

In [22]:
# Apply missingness indicator for binary features to all datasets
clean_train_df = add_binary_missing_indicators(clean_train_df)
clean_validation_df = add_binary_missing_indicators(clean_validation_df)
clean_test_df = add_binary_missing_indicators(clean_test_df)

### Categorical Feature Cleaning

Standardize `City`, `CountyOrParish`, and `PostalCode` by converting values to strings, removing surrounding whitespace, and replacing missing values with `"Unknown"`.

Target encoding will be performed within the modeling pipeline. The encoding approach is provisional and subject to further team review.

In [23]:
# Standardize categorical features for target encoding
clean_train_df = clean_categorical_features(clean_train_df)
clean_validation_df = clean_categorical_features(clean_validation_df)
clean_test_df = clean_categorical_features(clean_test_df)

## Preprocessing Pipeline

Define preprocessing steps for numerical, binary, and categorical features using sklearn pipelines. Missing values are imputed using training-derived statistics, numerical features are standardized, and categorical features are target-encoded.

Learned preprocessing parameters within the modeling pipeline are fitted separately for each cross-validation training fold to reduce data leakage. Fixed data-cleaning rules and training-derived ClosePrice thresholds are applied before cross-validation

In [24]:
# define the columns used by each pipeline.
area_columns = ["LivingArea", "LotSizeSquareFeet"]
property_columns = ["BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces", "ParkingTotal"]
coordinate_columns = ["Latitude", "Longitude"]
age_columns = ["PropertyAgeAtSale"]
binary_columns = ["AttachedGarageYN", "FireplaceYN", "NewConstructionYN", "PoolPrivateYN", "ViewYN"]
binary_missing_columns = [f"{col}_Missing" for col in binary_columns]
categorical_columns = ["City", "CountyOrParish","PostalCode"]

In [25]:
# Impute, log-transform, and standardize area features
area_pipeline = Pipeline([("imputation", SimpleImputer(strategy="median")),
                          ("log", FunctionTransformer(np.log1p)),
                          ("scaling", StandardScaler())])

# Impute property counts using mode, then standardize
property_pipeline = Pipeline([("imputation", SimpleImputer(strategy="most_frequent")),
                              ("scaling", StandardScaler())])

# Impute and standardize property age
property_age_pipeline = Pipeline([("imputation", SimpleImputer(strategy="median")),
                                  ("scaling", StandardScaler())])

# Standardize coordinates after geographic imputation
coordinate_pipeline = Pipeline([("scaling", StandardScaler())])

# Impute binary features using mode
binary_pipeline = Pipeline([("imputation", SimpleImputer(strategy="most_frequent"))])

# Encode high-cardinality categorical features
categorical_pipeline = Pipeline([("target_encoding", TargetEncoder(target_type="continuous", cv=5,\
                                                                   shuffle=True, random_state=42))])

In [26]:
# Build the ColumnTransformer by assigning each pipeline to its corresponding features.
preprocessor = ColumnTransformer([("area", area_pipeline, area_columns),
                                  ("property", property_pipeline, property_columns),
                                  ("coordinates", coordinate_pipeline, coordinate_columns),
                                  ("property_age", property_age_pipeline, age_columns),
                                  ("binary", binary_pipeline, binary_columns),
                                  ("binary_missing", "passthrough", binary_missing_columns),
                                  ("categorical", categorical_pipeline, categorical_columns)])

In [27]:
# Build the complete modeling pipeline
default_tree_pipeline = Pipeline([("coordinate_imputation", CoordinateImputer()),
                           ("preprocessing", preprocessor),
                           ("model", DecisionTreeRegressor(random_state=42))])

In [28]:
# Build the complete modeling pipeline
decision_tree_pipeline = Pipeline([("coordinate_imputation", CoordinateImputer()),
                           ("preprocessing", preprocessor),
                           ("model", DecisionTreeRegressor(random_state=42, max_depth=15, min_samples_leaf=10))])

In [29]:
# Build the Random Forest modeling pipeline
random_forest_pipeline = Pipeline([("coordinate_imputation", CoordinateImputer()),
                                   ("preprocessing", preprocessor),
                                   ("model", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])

In [30]:
# Prepare features and target 
model_features = (area_columns + property_columns + coordinate_columns +\
                  age_columns + binary_columns + binary_missing_columns + categorical_columns)

X_train = clean_train_df[model_features].copy()
y_train = clean_train_df["LogClosePrice"]

X_val = clean_validation_df[model_features].copy()
y_val = clean_validation_df["LogClosePrice"]

### Month-Based Cross-Validation

Use expanding-window cross-validation to evaluate the default and regularized Decision Tree models on chronological monthly splits.

Each fold trains on all preceding months and validates on the following month. The entire preprocessing pipeline is fitted independently within each training fold to prevent data leakage.

In [31]:
# Sort training records chronologically by source month
train_sorted = clean_train_df.sort_values("SourceMonth").reset_index(drop=True)

# Identify available training months
months = sorted(train_sorted["SourceMonth"].unique())

# Use the last five months for validation
validation_months = months[-5:]

cv_splits = []

for validation_month in validation_months:

    # Train on all months before the validation month
    train_idx = np.where(train_sorted["SourceMonth"] < validation_month)[0]

    # Validate on the selected month
    val_idx = np.where(train_sorted["SourceMonth"] == validation_month)[0]

    cv_splits.append((train_idx, val_idx))

In [32]:
# Calculate Mean Absolute Percentage Error
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Calculate Median Absolute Percentage Error
def mdape(y_true, y_pred):
    return np.median(np.abs((y_true - y_pred) / y_true)) * 100

In [33]:
# Custom scorer (calculate R² on the original ClosePrice scale)
def close_price_r2(estimator, X, y):
    predictions_log = estimator.predict(X)

    actual_prices = np.expm1(y)
    predicted_prices = np.expm1(predictions_log)

    return r2_score(actual_prices, predicted_prices)

def dollar_mape(estimator, X, y):
    return mape(np.expm1(y), np.expm1(estimator.predict(X)))

def dollar_mdape(estimator, X, y):
    return mdape(np.expm1(y), np.expm1(estimator.predict(X)))

### Month-Based Cross-Validation Results

Random Forest achieved the highest mean cross-validation R² of `0.8862`, with a MAPE of `11.68%` and an MdAPE of `7.74%` across five monthly validation folds.

The regularized Decision Tree also improved upon the baseline Linear Regression model, achieving a mean R² of `0.8525`, compared with `0.8003` for Linear Regression.

The default Decision Tree performed less consistently, with a mean R² of `0.7872`, highlighting the importance of controlling tree complexity.

Overall, Random Forest demonstrated the strongest cross-validation performance, suggesting that ensemble learning and the ability to capture nonlinear relationships improve prediction accuracy for residential property closing prices.

In [34]:
# Prepare chronologically sorted features and target for cross-validation
X_cv = train_sorted[model_features].copy()
y_cv = train_sorted["LogClosePrice"]

# Compare default and regularized Decision Trees
cv_results = []

for name, pipeline in [("Default Decision Tree", default_tree_pipeline), ("Regularized Decision Tree", decision_tree_pipeline),\
                       ("Random Forest", random_forest_pipeline)]:
    scores = cross_validate(pipeline, X_cv, y_cv, cv=cv_splits,
                            scoring={"R²": close_price_r2, "MAPE": dollar_mape,"MdAPE": dollar_mdape}, n_jobs=-1)

    cv_results.append({"Model": name, "Mean CV R²": scores["test_R²"].mean(),
        "Mean CV MAPE (%)": scores["test_MAPE"].mean(), "Mean CV MdAPE (%)": scores["test_MdAPE"].mean()})

cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df)

,Model,Mean CV R²,Mean CV MAPE (%),Mean CV MdAPE (%)
0,Default Decision Tree,0.787164,16.339647,10.561578
1,Regularized Decision Tree,0.852507,13.823096,9.334261
2,Random Forest,0.886214,11.678247,7.736890


In [35]:
# Display training and validation periods for each fold
for i, (train_idx, val_idx) in enumerate(cv_splits, start=1):

    train_months = train_sorted.iloc[train_idx]["SourceMonth"]
    val_months = train_sorted.iloc[val_idx]["SourceMonth"]

    print(f"Fold {i}: "f"Train {train_months.min()} to {train_months.max()} | "
          f"Validate {val_months.iloc[0]}")

Fold 1: Train 2024-01 to 2025-09 | Validate 2025-10
Fold 2: Train 2024-01 to 2025-10 | Validate 2025-11
Fold 3: Train 2024-01 to 2025-11 | Validate 2025-12
Fold 4: Train 2024-01 to 2025-12 | Validate 2026-01
Fold 5: Train 2024-01 to 2026-01 | Validate 2026-02


### Model Fitting and Validation Results

The default Decision Tree Regressor achieved a training R² of `0.9575` and a validation R² of `0.7886`, indicating overfitting. After limiting tree depth and increasing the minimum samples per leaf, the regularized Decision Tree achieved a training R² of `0.9036` and a validation R² of `0.8625`. The smaller gap suggests reduced overfitting and improved generalization.

Random Forest achieved the highest validation R² of `0.8885`, with a MAPE of `11.30%` and an MdAPE of `7.67%`. Although its training R² of `0.9683` suggests some overfitting, its strong validation performance and consistent cross-validation results indicate better predictive accuracy than both Decision Tree configurations. Overall, Random Forest demonstrated the strongest predictive performance among the evaluated models.

In [36]:
# Fit and evaluate all models on training and validation datasets
validation_results = []

for name, pipeline in [
    ("Default Decision Tree", default_tree_pipeline),
    ("Regularized Decision Tree", decision_tree_pipeline),
    ("Random Forest", random_forest_pipeline)]:
    
    # Fit the complete pipeline on the full training dataset
    pipeline.fit(X_train, y_train)

    # Generate predictions on the original ClosePrice scale
    train_predictions = np.expm1(pipeline.predict(X_train))
    val_predictions = np.expm1(pipeline.predict(X_val))

    # Calculate evaluation metrics
    train_r2 = r2_score(np.expm1(y_train), train_predictions)
    val_r2 = r2_score(np.expm1(y_val), val_predictions)
    val_mape = mape(np.expm1(y_val), val_predictions)
    val_mdape = mdape(np.expm1(y_val), val_predictions)

    # Store model results
    validation_results.append({"Model": name, "Training R²": train_r2, "Validation R²": val_r2,
                               "Validation MAPE (%)": val_mape, "Validation MdAPE (%)": val_mdape})

# Create comparison DF
validation_results_df = pd.DataFrame(validation_results)

display(validation_results_df.round(4))

,Model,Training R²,Validation R²,Validation MAPE (%),Validation MdAPE (%)
0,Default Decision Tree,0.9575,0.7886,16.0424,10.4679
1,Regularized Decision Tree,0.9036,0.8625,13.4179,9.2788
2,Random Forest,0.9683,0.8885,11.3036,7.6687


In [38]:
# Add Linear Regression baseline validation results
baseline_validation = pd.DataFrame([{"Model": "Linear Regression", "Training R²": 0.8248, 
                                     "Validation R²": 0.796856, "Validation MAPE (%)": 15.812493,
                                     "Validation MdAPE (%)": 11.949841}])

# Combine baseline with advanced model results
validation_comparison_df = pd.concat(baseline_validation, validation_results_df],ignore_index=True)

display(validation_comparison_df)

,Model,Training R²,Validation R²,Validation MAPE (%),Validation MdAPE (%)
0,Linear Regression,0.824800,0.796856,15.812493,11.949841
1,Default Decision Tree,0.957509,0.788616,16.042402,10.467852
2,Regularized Decision Tree,0.903643,0.862493,13.417920,9.278814
3,Random Forest,0.968265,0.888498,11.303567,7.668708


### April 2026 Test Results

The three tree-based models were evaluated on the held-out April 2026 test dataset and compared against the Linear Regression baseline. The default Decision Tree achieved a test R² of `0.8048`, slightly below the Linear Regression baseline of `0.8093`. The regularized Decision Tree improved test performance to `0.8548`, demonstrating that limiting tree complexity helps reduce overfitting and improve generalization.

Random Forest achieved the highest test R² of `0.8893`, with a MAPE of `11.55%` and an MdAPE of `7.78%`. Its test performance was consistent with the month-based cross-validation and March 2026 validation results. Overall, Random Forest demonstrated the strongest predictive performance outperforming the Linear Regression baseline and both Decision Tree configurations on the held-out test dataset.

In [39]:
# Prepare the April 2026 test dataset
X_test = clean_test_df[model_features].copy()
y_test = clean_test_df["LogClosePrice"]

# Convert actual values to the original ClosePrice scale
y_test_actual = np.expm1(y_test)

# Store test evaluation results
test_results = []

for name, pipeline in [("Default Decision Tree", default_tree_pipeline),
                       ("Regularized Decision Tree", decision_tree_pipeline),
                       ("Random Forest", random_forest_pipeline)]:
    
    # Predict ClosePrice for the April 2026 test dataset
    test_predictions_log = pipeline.predict(X_test)

    # Convert predictions to the original dollar scale
    test_predictions = np.expm1(test_predictions_log)

    # Calculate test metrics in dallor scale
    test_r2 = r2_score(y_test_actual, test_predictions)
    test_mape = mape(y_test_actual, test_predictions)
    test_mdape = mdape(y_test_actual, test_predictions)

    # Store the results
    test_results.append({"Model": name, "Test R²": test_r2, "Test MAPE (%)": test_mape,
                         "Test MdAPE (%)": test_mdape})

test_results_df = pd.DataFrame(test_results)

display(test_results_df)

,Model,Test R²,Test MAPE (%),Test MdAPE (%)
0,Default Decision Tree,0.804780,15.914431,10.389610
1,Regularized Decision Tree,0.854771,13.763280,9.448494
2,Random Forest,0.889316,11.554174,7.775205


### Model Behavior (Strengths/Weaknesses)

**Default Decision Tree:** The model achieved a high training R² of `0.9575` but a considerably lower validation R² of `0.7886`, indicating overfitting. Although Decision Trees can capture nonlinear relationships and are relatively interpretable, unrestricted tree growth can reduce generalization performance.

**Regularized Decision Tree:** Limiting the maximum depth to 15 and requiring at least 10 samples per leaf reduced overfitting. The model achieved a training R² of `0.9036` and a validation R² of `0.8625`, demonstrating improved generalization.

**Random Forest:** The model achieved the highest validation of R² `0.8885` and test of R² `0.8893`. By averaging predictions across multiple trees, Random Forest reduces the variance associated with individual decision trees. However, it is more computationally expensive and its higher training R² of `0.9683` also indicates some remaining overfitting.

In [40]:
baseline_test = pd.DataFrame([{"Model": "Linear Regression", "Test R²": 0.8093,
                               "Test MAPE (%)": 15.8622, "Test MdAPE (%)": 11.8900}])

test_comparison_df = pd.concat([baseline_test, test_results_df], ignore_index=True)
display(test_comparison_df)

,Model,Test R²,Test MAPE (%),Test MdAPE (%)
0,Linear Regression,0.809300,15.862200,11.890000
1,Default Decision Tree,0.804780,15.914431,10.389610
2,Regularized Decision Tree,0.854771,13.763280,9.448494
3,Random Forest,0.889316,11.554174,7.775205
